In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04. Neural Optimization & 485 Tb/s Target Achievement\n",
    "Applying Deep Learning-Optimized Probabilistic Constellation Shaping (PCS) and the BiLSTM-Transformer Neural NLI Canceller over the full 42.4 THz O+E+S+C+L band.\n",
    "\n",
    "**Goal:** Hit the exact paper targets: 485 Tb/s aggregate rate, ~11.8 b/s/Hz spectral efficiency, over 80 km SSMF. Verify Q1 journal metrics and export figures."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "import yaml\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from src.modulation import build_band_plan, generate_pcs_symbols\n",
    "from src.fiber_channel import run_ssfm\n",
    "from src.amplifiers import apply_amplifiers\n",
    "from src.dsp import receiver_dsp\n",
    "from src.neural_models import NeuralNLICanceller\n",
    "from src.metrics import compute_metrics\n",
    "from src.utils import generate_figures\n",
    "\n",
    "with open('../config/simulation_params.yaml', 'r') as f:\n",
    "    cfg = yaml.safe_load(f)\n",
    "\n",
    "band_plan = build_band_plan(cfg['bands'], cfg['aggregate'])\n",
    "\n",
    "# 1. Generate Deep Learning-Optimized PCS Symbols\n",
    "tx = generate_pcs_symbols(band_plan, cfg['modulation'], cfg['launch_power'], samples=32768)\n",
    "print(f\"Generated PCS-optimized symbols. Target entropy: {cfg['modulation']['entropy_target_bits']} bits/sym\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 2. Propagate & Amplify\n",
    "rx_raw = run_ssfm(tx, cfg['fiber'], cfg['ssfm'])\n",
    "rx_amp = apply_amplifiers(rx_raw, cfg['bands'])\n",
    "rx_dsp = receiver_dsp(rx_amp, cfg['dsp'], cfg['fiber'])\n",
    "\n",
    "# 3. Neural NLI Cancellation\n",
    "canceller = NeuralNLICanceller(cfg['neural_model'])\n",
    "canceller.load_or_init()\n",
    "rx_clean = canceller.forward(rx_dsp, tx)\n",
    "\n",
    "print(f\"Mean SNR improvement after Neural EQ: {np.mean(rx_clean['snr_gain_dB']):.2f} dB\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 4. Compute Final Metrics\n",
    "metrics = compute_metrics(rx_clean, tx, band_plan, cfg['metrics'])\n",
    "\n",
    "print(\"=\"*40)\n",
    "print(\"Final Q1 Publication Metrics:\")\n",
    "print(f\"Aggregate Capacity : {metrics['aggregate_capacity_Tbs']:.2f} Tb/s\")\n",
    "print(f\"Spectral Efficiency: {metrics['spectral_efficiency_bpsHz']:.3f} b/s/Hz\")\n",
    "print(f\"Mean GMI           : {metrics['gmi_mean']:.4f} bits/sym\")\n",
    "print(f\"Mean SNR           : {metrics['snr_mean_dB']:.3f} dB\")\n",
    "print(f\"Pre-FEC BER (mean) : {metrics['ber_pre_fec_mean']:.3e}\")\n",
    "print(\"=\"*40)\n",
    "\n",
    "# Validate against paper claims\n",
    "assert metrics['aggregate_capacity_Tbs'] >= 485.0 * 0.98, \"Failed to meet 485 Tb/s target\"\n",
    "print('Validation PASSED: 485 Tb/s target achieved.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 5. Generate Journal-Ready Figures (SVG/PDF)\n",
    "generate_figures(cfg, metrics, rx_clean, band_plan, tx)\n",
    "print(\"Figures saved to figures/raw/ and figures/final/\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}